# 02b · CLIP probe for fully AI-generated faces

The CNNs (02a) learn face-swap and re-enactment traces; they have never seen a **fully generated** face. The
CLIP probe covers that: a frozen **CLIP ViT-L/14** image encoder (trained on 400M image–text pairs, never on
deepfakes) with a **logistic-regression** head (Ojha et al., CVPR 2023, "UnivFD"). Because the encoder is
frozen, the probe generalises to generators it has not seen much better than a fine-tuned network.

- **Training data:** real faces from every dataset vs. StyleGAN + diffusion fakes **and** FF++ / Celeb-DF fakes
  (at most `MAX_PER_SET` images per dataset). Each training image is encoded twice: clean and degraded
  (blur / resample / JPEG / noise), so the probe doesn't rely on image quality either.
- `C` (regularisation) is chosen on validation AUC.
- The validation list is the same as 02a's (`VAL_MAX_PER_SET` must match), so 02c can fuse CNN and CLIP scores
  image by image.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 · **Internet:** On (downloads the CLIP weights, ~1.7 GB)
- **Inputs:** outputs of **01a**, **01b** and **01c**

**Produces:** `/kaggle/working/outputs/clip/` (`probe.pt`, `val_preds.npz`, `probe_report.json`).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup, find_datasets
setup(data=False)

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

SETS = {k: v for k, v in find_datasets().items() if k != "DFDC"}   # DFDC is test-only
print("datasets:", SETS)
assert {"StyleGAN", "Diffusion"} & set(SETS), "Attach 01c's output: the probe needs StyleGAN / diffusion fakes"
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## Settings

In [ ]:
MAX_PER_SET = 15000       # training images per dataset (x2 views)
VAL_MAX_PER_SET = 5000    # must equal 02a's VAL_MAX_PER_SET
KEEP_TRAIN_FEATURES = False   # the cached training features are large; keep them only to re-fit the probe later

## Encode + fit the probe

In [ ]:
args = " ".join(f'--data "{k}={v}"' for k, v in SETS.items())
!TQDM_MININTERVAL=60 python -m src.classifier.clip_probe --out /kaggle/working/outputs/clip --max-per-set {MAX_PER_SET} --val-max-per-set {VAL_MAX_PER_SET} --workers 4 {args}

## Validation results

In [ ]:
OUT = Path("/kaggle/working/outputs/clip")
report = json.loads((OUT / "probe_report.json").read_text())
print(f"C = {report['C']}, pooled validation AUC = {report['val_auc']:.4f}")
display(pd.Series(report["per_dataset_auc"], name="AUC (two-class sets)").to_frame())

z = np.load(OUT / "val_preds.npz")
df = pd.DataFrame({"dataset": z["datasets"], "label": z["labels"], "logit": z["logits"]})
fig, ax = plt.subplots(figsize=(9, 3.6))
for (name, label), g in df.groupby(["dataset", "label"]):
    ax.hist(g.logit, bins=50, histtype="step", label=f"{name} {'fake' if label else 'real'}")
ax.axvline(0, color="gray", ls="--"); ax.set_xlabel("CLIP probe logit"); ax.legend(fontsize=7, ncol=2)
ax.set_title("Validation score distributions (fake > 0)")
plt.tight_layout(); plt.show()
if not KEEP_TRAIN_FEATURES:
    (OUT / "feats_train.npz").unlink(missing_ok=True)
!du -sh /kaggle/working/outputs/clip